### IFN647 Week 12 Review Question 4 code - Text Contrastive Learning
<author> &copy; Prepared by Professor Yuefeng Li </author>


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

# --------------------------------
# 1. A small text dataset (INPUT)
# --------------------------------

sentences = [
    "a dog is running in the park",
    "a cat is sleeping on the sofa",
    "a man is riding a bicycle",
    "the weather is very sunny today",
    "deep learning models are powerful",
    "machine learning for NLP",
    "Deep learning models are inspired by the human brain",
    "Sentiment Analysis Determines emotions or opinions in text"
]

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

# save the text senetences in a dataset
dataset = TextDataset(sentences)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [16]:
# -----------------------------------------
# 2. Augmentation (POSITIVE PAIR CREATION)
# ----------------------------------------
def dropout_noise(text, drop_prob=0.2):      # Each word is dropped independently with probability 0.2.
    words = text.split()
    words = [w for w in words if torch.rand(1).item() > drop_prob]   # torch.rand(1).item() = The value is sampled from Uniform(0, 1)
    return " ".join(words) if len(words) > 2 else text

def create_pair(text):
    return text, dropout_noise(text)

In [18]:
# ----------------------------
# 3. Encoder model (BERT)
# ----------------------------
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder.to(device)

def encode(texts):
    tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    tokens = {k: v.to(device) for k, v in tokens.items()}

    outputs = encoder(**tokens)
    cls_emb = outputs.last_hidden_state[:, 0]  # [CLS] token - start of sequence
    return cls_emb

# [CLS] token acts as a summary of the sentence, and has seen all tokens via self-attention
# The encoder output of [CLS] can represent the whole sentence, but It may not capture semantic similarity as well as other methods

In [20]:
# ----------------------------
# 4. Contrastive loss (InfoNCE)
# ----------------------------
def contrastive_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    batch_size = z1.size(0)
    
    z = torch.cat([z1, z2], dim=0)  # 2N x d

    sim = torch.matmul(z, z.T) / temperature

    labels = torch.arange(batch_size).to(device)
    labels = torch.cat([labels, labels], dim=0)

    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(device)
    sim = sim.masked_fill(mask, -1e9)

    positives = torch.cat([
        torch.diag(sim, batch_size),
        torch.diag(sim, -batch_size)
    ])

    loss = -positives + torch.logsumexp(sim, dim=1)
    return loss.mean()

In [24]:
# ----------------------------
# 5. Training loop (INPUT → OUTPUT pipeline)
# ----------------------------
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-5)

for epoch in range(3):
    total_loss = 0

    for batch in loader:
        texts = list(batch)

        # ---- INPUT: create positive pairs ----
        x1, x2 = zip(*[create_pair(t) for t in texts])
        print("x1: ")
        print(x1)
        print("x2: ")
        print(x2)

        # ---- ENCODING STEP ----
        z1 = encode(list(x1))
        z2 = encode(list(x2))
        
        # ---- LOSS COMPUTATION ----
        loss = contrastive_loss(z1, z2)
        print("Initial loss value: ")
        print(loss)

        # gradient-based learning
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

x1: 
('a man is riding a bicycle', 'deep learning models are powerful', 'Sentiment Analysis Determines emotions or opinions in text', 'machine learning for NLP')
x2: 
('a man is riding a bicycle', 'deep learning models are', 'Sentiment Analysis Determines emotions or opinions in', 'machine learning for NLP')
Initial loss value: 
tensor(1.5132, grad_fn=<MeanBackward0>)
x1: 
('a dog is running in the park', 'Deep learning models are inspired by the human brain', 'a cat is sleeping on the sofa', 'the weather is very sunny today')
x2: 
('a dog is in the park', 'Deep models inspired by the human', 'a is sleeping sofa', 'the weather very sunny')
Initial loss value: 
tensor(1.5717, grad_fn=<MeanBackward0>)
Epoch 1, Loss: 3.0849
x1: 
('the weather is very sunny today', 'Deep learning models are inspired by the human brain', 'a dog is running in the park', 'Sentiment Analysis Determines emotions or opinions in text')
x2: 
('weather is very sunny today', 'Deep learning models are inspired by the

In [8]:

# ----------------------------
# 6. OUTPUT: final embeddings
# ----------------------------
def get_embedding(text):
    encoder.eval()
    with torch.no_grad():
        return encode([text])[0].cpu()

print("\nExample embeddings:")
#print(get_embedding("a dog is running"))
print(get_embedding("Deep learning is a branch of machine learning within artificial intelligence"))


Example embeddings:
tensor([-1.4184e-02, -4.3037e-01, -7.4423e-01,  2.2734e-01, -4.6556e-02,
        -3.9401e-01,  4.3681e-01,  4.9231e-02,  6.0980e-01, -4.2770e-01,
         1.4248e-01, -1.3526e-01, -4.2431e-01, -1.5601e-02, -8.4692e-01,
        -7.5507e-01,  3.2792e-01, -1.8497e-01,  1.6848e-01,  6.0186e-01,
        -1.6908e-02, -2.1222e-01, -1.4589e-01,  7.0421e-01,  4.7256e-01,
        -2.4664e-01, -1.1534e-01, -1.7495e-01,  2.0750e-01,  7.5771e-02,
        -5.9213e-01,  1.0791e-01, -3.6300e-01, -1.3759e-01,  1.7158e-01,
        -1.3757e-01, -5.0678e-01, -3.8455e-01, -3.9712e-01,  1.4280e-01,
        -7.7947e-01, -4.1958e-03, -3.8138e-01,  2.1572e-01, -5.4461e-01,
         5.8012e-02, -2.5522e+00,  5.5374e-01, -3.4524e-01,  3.4245e-01,
        -3.6769e-01,  3.5690e-01, -4.6474e-01,  5.0309e-01,  6.1956e-01,
         1.6815e-01,  5.8726e-01, -7.4648e-01, -2.6199e-01, -2.5064e-01,
         3.4386e-01,  6.9648e-01, -2.8332e-01, -3.0164e-01,  4.3656e-01,
        -2.6200e-01, -1.9443e-